# Stage 2: particle filter + guarded trend blend

Run `00_colab_setup.ipynb` first, then run this notebook in the same Colab runtime. It pulls the latest repository, installs the locked Numba dependency, runs an eight-well smoke test, and then evaluates the promoted 75% two-batch particle-filter + 25% Stage 1 blend on all 773 wells. A CPU runtime is sufficient; the full run is expected to take roughly 30--45 minutes.

In [ ]:
from pathlib import Path
import json
import os
import subprocess

repo_dir = Path('/content/ROGII')
data_dir = Path('/content/rogii-data')
artifact_dir = Path('/content/drive/MyDrive/kaggle/rogii/artifacts')
assert (repo_dir / '.git').is_dir(), 'Run 00_colab_setup.ipynb first'
assert (data_dir / 'train').is_dir(), 'Competition data is not copied to /content yet'
os.environ['ROGII_DATA_DIR'] = str(data_dir)
os.environ['ROGII_ARTIFACT_DIR'] = str(artifact_dir)

In [ ]:
subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)

In [ ]:
SMOKE_RUN_ID = 'stage2_pf_trend_blend_smoke_v001'
smoke_dir = artifact_dir / SMOKE_RUN_ID
if not (smoke_dir / 'metrics.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-train',
        '--config', 'configs/experiment/pf_trend_blend.yaml',
        '--run-id', SMOKE_RUN_ID,
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--limit-wells', '8',
    ], cwd=repo_dir, check=True)
json.loads((smoke_dir / 'metrics.json').read_text())

In [ ]:
RUN_ID = 'stage2_pf_trend_blend_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'metrics.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-train',
        '--config', 'configs/experiment/pf_trend_blend.yaml',
        '--run-id', RUN_ID,
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
    ], cwd=repo_dir, check=True)
candidate_metrics = json.loads((run_dir / 'metrics.json').read_text())
candidate_metrics

In [ ]:
stage1_dir = artifact_dir / 'stage1_trend_blend_full_v001'
if (stage1_dir / 'metrics.json').is_file():
    stage1_metrics = json.loads((stage1_dir / 'metrics.json').read_text())
    print({
        'stage1_rmse': stage1_metrics['pooled_rmse'],
        'stage2_rmse': candidate_metrics['pooled_rmse'],
        'rmse_delta': candidate_metrics['pooled_rmse'] - stage1_metrics['pooled_rmse'],
    })
else:
    print('Stage 1 artifact not found; Stage 2 itself completed successfully.')

In [ ]:
if (stage1_dir / 'metrics.json').is_file():
    comparison_dir = artifact_dir / 'stage2_vs_stage1_v001'
    if not (comparison_dir / 'comparison.json').is_file():
        subprocess.run([
            'uv', 'run', 'rogii-compare',
            '--baseline', str(stage1_dir),
            '--candidate', str(run_dir),
            '--output-dir', str(comparison_dir),
        ], cwd=repo_dir, check=True)